# A Practical Guide for Single-Cell Data Analysis in Neurosciences

## What You'll Learn
- Complete scRNA-seq analysis pipeline from raw data to biological insights
- Parameter optimization strategies for each analysis step
- Integration with pathway analysis and functional annotation tools
- Performance optimization using GPU acceleration
- Troubleshooting common issues

## Project Configuration

In [ ]:
# Set up main working directory
import os
working_dir = "/media/mim/98f13536-4fbf-47cf-a5eb-647479dcaef1/Computational-Neuroscience-other-Phd-Stuufs/BRAIN_PROGRAM_CONTENTS/Neurogenomics-Codings/sc-neuro"
os.makedirs(working_dir, exist_ok=True)
os.chdir(working_dir)

In [ ]:
# Define folder structure
folders = [
    "raw_data",
    "processed_data",
    "results",
    "results/figures",
    "results/tables"
]

# Create folders
for folder in folders:
    path = os.path.join(working_dir, folder)
    os.makedirs(path, exist_ok=True)

In [ ]:
# Set Save paths for easy use later
raw_data_dir = os.path.join(working_dir, "raw_data")
processed_data_dir = os.path.join(working_dir, "processed_data")
results_dir = os.path.join(working_dir, "results")
figures_dir = os.path.join(working_dir, "results/figures")
tables_dir = os.path.join(working_dir, "results/tables")

## Environment Setup

In [ ]:
%%capture
!pip install scanpy
!pip install pandas numpy matplotlib seaborn
!pip install leidenalg python-igraph
!pip install bbknn scvelo cellrank
!pip install decoupler-py gseapy

## GPU Support Installation (Optional but Recommended)


In [ ]:
!rocm-smi

##### Install CuPy (GPU array library, matches Colab CUDA runtime)

In [ ]:
%%capture
!pip install cupy-cuda11x

# RAPIDS SingleCell (still experimental in Colab, but works with recent CUDA runtimes)
!pip install rapids-singlecell

## Additional Analysis Tools

In [ ]:
%%capture
# Pathway & regulatory analysis
!pip install gseapy pyscenic
!pip install celltypist scarches

# Visualization
!pip install plotly nbformat>=4.2.0

# Data integration
!pip install harmony-pytorch scanorama

In [ ]:
# Ignore all warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Imports
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy

In [ ]:
# Set the SCIPY_ARRAY_API environment variable
os.environ["SCIPY_ARRAY_API"] = "1"

# Configure scanpy settings
sc.settings.verbosity = 3  # Verbosity level
sc.settings.set_figure_params(dpi=80, facecolor='white')
sc.settings.n_jobs = -1  # Use all available cores

# Memory optimization
sc.settings.max_memory = 50  # GB, adjust based on your system

In [ ]:
# GPU configuration (if available)
# The following import caused an AttributeError due to a potential circular import in cuml.
# Temporarily disabling to allow the notebook to proceed.
try:
    import rapids_singlecell as rsc
    import cupy as cp
    GPU_AVAILABLE = True
    print("GPU acceleration enabled!")
except ImportError:
     GPU_AVAILABLE = False
     print("Running on CPU")
GPU_AVAILABLE = False # Set to False as GPU import is skipped.
print("Running on CPU")

In [ ]:
# Set up plotting parameters
import matplotlib
matplotlib.rcParams['figure.figsize'] = (4, 4)
matplotlib.rcParams['pdf.fonttype'] = 42  # For publication-quality figures

## Data Acquisition


## Downloading Public Datasets

In [ ]:
sc.__version__

## GEO Datasets

In [ ]:
# Method 1: Using scanpy built-in datasets
# show all built-in datasets in scanpy
list(sc.datasets.__dict__.keys())

In [ ]:
# load built-in data
adata = sc.datasets.pbmc3k()  # 3k PBMCs from 10X Genomics

In [ ]:
adata

In [ ]:
%%capture
!pip install GEOparse

In [ ]:
# Method 2: Download from GEO
import GEOparse
import urllib.request

def download_geo_data(geo_id, output_dir=raw_data_dir):
    """
    Download data from GEO database

    Parameters:
    -----------
    geo_id : str
        GEO accession number (e.g., 'GSE123456')
    """
    import os
    os.makedirs(output_dir, exist_ok=True)

    gse = GEOparse.get_GEO(geo=geo_id, destdir=output_dir)

    # Get supplementary files
    for gsm_name, gsm in gse.gsms.items():
        # Check if supplementary_file_url attribute exists before accessing it
        if hasattr(gsm, 'supplementary_file_url'):
            for supp_file in gsm.supplementary_file_url:
                try:
                    urllib.request.urlretrieve(supp_file,
                                               f"{output_dir}/{gsm_name}_{supp_file.split('/')[-1]}")
                except Exception as e:
                    print(f"Error downloading supplementary file {supp_file}: {e}")
        else:
            print(f"GSM {gsm_name} does not have the attribute 'supplementary_file_url'. Available attributes are: {dir(gsm)}")

    return gse

In [ ]:
# Example usage
gse = download_geo_data('GSE123456')

In [ ]:
gse

## 10X Genomics Datasets

In [ ]:
# Method 3: Loading 10X Genomics Data
def load_10x_data(data_path, min_genes=200, min_cells=3):
    """
    Load 10X Genomics data with initial filtering

    Parameters:
    -----------
    data_path : str
        Path to the directory containing matrix.mtx, genes.tsv, barcodes.tsv
    min_genes : int
        Minimum number of genes expressed per cell
    min_cells : int
        Minimum number of cells expressing a gene
    """
    # Read 10X data
    adata = sc.read_10x_h5(
        data_path
    )

    # Make variable names unique
    adata.var_names_make_unique()

    # Initial filtering
    sc.pp.filter_cells(adata, min_genes=min_genes)
    sc.pp.filter_genes(adata, min_cells=min_cells)

    print(f"Loaded dataset: {adata.shape[0]} cells × {adata.shape[1]} genes")

    return adata

In [ ]:
os.listdir(raw_data_dir)

In [ ]:
# Example usage
# load one sample data from raw_data_dir using load_10x_data
adata = load_10x_data(os.path.join(raw_data_dir, 'GSM4635080_P1_S1_filtered_gene_bc_matrices_h5.h5'))

In [ ]:
adata

## Loading Multiple Samples

In [ ]:
def load_multiple_samples(sample_dict, batch_key='batch'):
    """
    Load and concatenate multiple samples

    Parameters:
    -----------
    sample_dict : dict
        Dictionary with sample names as keys and paths as values
    batch_key : str
        Key to store batch information
    """
    adatas = {}

    for sample_name, path in sample_dict.items():
        print(f"Loading {sample_name}...")
        # Using the updated load_10x_data function which handles both .h5 and mtx
        adata = load_10x_data(path)
        adata.obs[batch_key] = sample_name
        adatas[sample_name] = adata

    # Concatenate all samples, using the dictionary keys for labels and storing them in batch_key column
    adata_concat = sc.concat(adatas, label=batch_key)

    print(f"Combined dataset: {adata_concat.shape[0]} cells × {adata_concat.shape[1]} genes")

    return adata_concat

In [ ]:
# Example usage
samples = {
    'S1_WT': os.path.join(raw_data_dir, 'GSM4635080_P1_S1_filtered_gene_bc_matrices_h5.h5'),
    'S2_Fefz2-KO': os.path.join(raw_data_dir, 'GSM4635087_Fezf2KO_P1_filtered_feature_bc_matrix.h5')
}
samples

In [ ]:
adata = load_multiple_samples(samples)

In [ ]:
adata

In [ ]:
adata.obs

## Quality Control


### Comprehensive QC Metrics

In [ ]:
import numpy as np
import scanpy as sc
import scipy
import matplotlib.pyplot as plt


def calculate_qc_metrics(
    adata,
    mt_prefix="mt-",
    rb_prefix="Rps|Rpl",
    hb_genes=["Hba-a1", "Hba-a2", "Hbb-bs"],
    min_genes=200,
    max_genes=2500,
    max_mt_percent=5,
):
    """
    Calculate and visualize comprehensive single-cell QC metrics.

    Gene naming conventions:
    -------------------------
    Mitochondrial genes:
        - Mouse: 'mt-'
        - Human: 'MT-'

    Ribosomal genes:
        - Mouse: 'Rps', 'Rpl'
        - Human: 'RPS', 'RPL'

    Hemoglobin genes:
        - Mouse: Hba-a1, Hba-a2, Hbb-bs
        - Human: HBA1, HBA2, HBB

    Parameters
    ----------
    adata : AnnData
        Annotated data matrix
    mt_prefix : str
        Prefix for mitochondrial genes
    rb_prefix : str
        Regex pattern for ribosomal genes
    hb_genes : list
        List of hemoglobin genes
    min_genes : int
        Minimum genes per cell (visual guide)
    max_genes : int
        Maximum genes per cell (visual guide)
    max_mt_percent : float
        Maximum mitochondrial percentage (visual guide)
    """

    # Identify gene categories
    adata.var["mt"] = adata.var_names.str.startswith(mt_prefix)
    adata.var["ribo"] = adata.var_names.str.contains(f"^({rb_prefix})")
    adata.var["hb"] = adata.var_names.isin(hb_genes)

    # Compute QC metrics
    sc.pp.calculate_qc_metrics(
        adata,
        qc_vars=["mt", "ribo", "hb"],
        percent_top=None,
        log1p=False,
        inplace=True,
    )

    # Additional metrics
    if scipy.sparse.issparse(adata.X):
        adata.obs["n_genes"] = (adata.X > 0).sum(axis=1).A1
    else:
        adata.obs["n_genes"] = (adata.X > 0).sum(axis=1)

    adata.obs["log10_total_counts"] = np.log10(adata.obs["total_counts"] + 1)

    # QC violin plots (Scanpy-managed)
    sc.pl.violin(
        adata,
        ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo"],
        jitter=0.4,
        multi_panel=True,
        show=True,
    )

    # Custom diagnostic plots
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))

    # Counts vs genes
    axes[0].scatter(
        adata.obs["total_counts"],
        adata.obs["n_genes_by_counts"],
        c=adata.obs["pct_counts_mt"],
        s=2,
        alpha=0.5,
    )
    axes[0].set_xlabel("Total counts")
    axes[0].set_ylabel("Genes per cell")

    # Gene count distribution
    axes[1].hist(adata.obs["n_genes_by_counts"], bins=50, alpha=0.7)
    axes[1].axvline(min_genes, color="red", linestyle="--")
    axes[1].axvline(max_genes, color="red", linestyle="--")
    axes[1].set_xlabel("Genes per cell")

    # Mitochondrial percentage
    axes[2].hist(adata.obs["pct_counts_mt"], bins=50, alpha=0.7)
    axes[2].axvline(max_mt_percent, color="red", linestyle="--")
    axes[2].set_xlabel("Mitochondrial %")

    # Library complexity
    axes[3].scatter(
        adata.obs["log10_total_counts"],
        adata.obs["n_genes_by_counts"] / adata.obs["total_counts"],
        s=2,
        alpha=0.5,
    )
    axes[3].set_xlabel("log10(total counts)")
    axes[3].set_ylabel("Genes per UMI")

    plt.tight_layout()
    plt.show()

    # Summary statistics
    print("\nQC summary:")
    print(f"Cells: {adata.n_obs}")
    print(f"Median genes per cell: {adata.obs['n_genes_by_counts'].median():.0f}")
    print(f"Median counts per cell: {adata.obs['total_counts'].median():.0f}")
    print(f"Median MT%: {adata.obs['pct_counts_mt'].median():.2f}")

    return adata

In [ ]:
# Apply QC metrics
adata = calculate_qc_metrics(adata)

## Adaptive Filtering

In [ ]:
def adaptive_filtering(adata,
                       mad_threshold=5,
                       min_genes=200,
                       min_counts=500):
    """
    Apply adaptive filtering based on MAD (Median Absolute Deviation)

    Parameters:
    -----------
    mad_threshold : float
        Number of MADs from median for outlier detection
    """
    from scipy import stats

    # Calculate MAD-based thresholds
    def mad_based_outlier(data, threshold=3):
        median = np.median(data)
        mad = stats.median_abs_deviation(data)
        lower = median - threshold * mad
        upper = median + threshold * mad
        return lower, upper

    # Calculate thresholds
    n_genes_lower, n_genes_upper = mad_based_outlier(adata.obs['n_genes_by_counts'], mad_threshold)
    counts_lower, counts_upper = mad_based_outlier(adata.obs['total_counts'], mad_threshold)
    mt_lower, mt_upper = mad_based_outlier(adata.obs['pct_counts_mt'], mad_threshold)

    # Apply minimum thresholds
    n_genes_lower = max(n_genes_lower, min_genes)
    counts_lower = max(counts_lower, min_counts)

    print(f"Adaptive thresholds:")
    print(f"  Genes: {n_genes_lower:.0f} - {n_genes_upper:.0f}")
    print(f"  Counts: {counts_lower:.0f} - {counts_upper:.0f}")
    print(f"  MT%: 0 - {mt_upper:.2f}")

    # Filter cells
    adata = adata[
        (adata.obs['n_genes_by_counts'] >= n_genes_lower) &
        (adata.obs['n_genes_by_counts'] <= n_genes_upper) &
        (adata.obs['total_counts'] >= counts_lower) &
        (adata.obs['total_counts'] <= counts_upper) &
        (adata.obs['pct_counts_mt'] < mt_upper)
    ].copy()

    print(f"Cells after filtering: {adata.n_obs}")

    return adata

In [ ]:
# Apply adaptive filtering
adata = adaptive_filtering(adata)

## Normalization and Feature Selection


### Normalization Methods

In [ ]:
def normalize_data(adata,
                  method='standard',
                  target_sum=1e4,
                  exclude_highly_expressed=True):
    """
    Normalize expression data with multiple method options

    Parameters:
    -----------
    method : str
        'standard', 'scran', or 'sctransform'
    target_sum : float
        Target sum for normalization
    exclude_highly_expressed : bool
        Whether to exclude highly expressed genes from normalization
    """

    # Save raw counts as a true copy
    adata.raw = adata.copy() # Changed to .copy()

    if method == 'standard':
        # Standard log-normalization
        sc.pp.normalize_total(adata, target_sum=target_sum,
                             exclude_highly_expressed=exclude_highly_expressed)
        sc.pp.log1p(adata)

    elif method == 'scran':
        # scran pooling-based size factor normalization
        import scanpy.external as sce

        # Preliminary clustering for size factor calculation
        adata_pp = adata.copy()
        sc.pp.normalize_total(adata_pp, target_sum=target_sum)
        sc.pp.log1p(adata_pp)
        sc.pp.pca(adata_pp, n_comps=15)
        sc.pp.neighbors(adata_pp)
        sc.tl.leiden(adata_pp, resolution=0.5)

        # Calculate size factors
        adata.obs['size_factors'] = sce.pp.scran_normalize(
            adata,
            clusters=adata_pp.obs['leiden'],
            return_size_factors=True
        )

        # Normalize with size factors
        adata.X /= adata.obs['size_factors'].values[:, None]
        sc.pp.log1p(adata)

    elif method == 'sctransform':
        # SCTransform normalization
        try:
            import scanpy.external as sce
            sce.pp.sctransform(adata)
        except ImportError:
            print("SCTransform not available, using standard normalization")
            sc.pp.normalize_total(adata, target_sum=target_sum)
            sc.pp.log1p(adata)

    print(f"Normalization complete using {method} method")
    return adata

In [ ]:
# Apply normalization(standard)
adata = normalize_data(adata, method='standard')

In [ ]:
adata

### Highly Variable Gene Selection

In [ ]:
def select_highly_variable_genes(adata,
                                 method='seurat',
                                 n_top_genes=2000,
                                 min_mean=0.0125,
                                 max_mean=3,
                                 min_disp=0.5,
                                 batch_key=None):
    """
    Select highly variable genes with different methods

    Parameters:
    -----------
    method : str
        'seurat', 'cell_ranger', or 'seurat_v3'
    n_top_genes : int
        Number of highly variable genes to select
    batch_key : str
        Key for batch correction in HVG selection
    """

    if batch_key:
        # Batch-aware HVG selection
        sc.pp.highly_variable_genes(
            adata,
            n_top_genes=n_top_genes,
            flavor=method,
            batch_key=batch_key,
            subset=False
        )
    else:
        # Standard HVG selection
        sc.pp.highly_variable_genes(
            adata,
            min_mean=min_mean,
            max_mean=max_mean,
            min_disp=min_disp,
            n_top_genes=n_top_genes,
            flavor=method,
            subset=False
        )

    # Plot HVG selection
    sc.pl.highly_variable_genes(adata)

    # Print statistics
    print(f"Number of highly variable genes: {adata.var['highly_variable'].sum()}")

    # Store full data and filter
    # The adata.raw attribute should be set ONCE, typically after filtering raw counts,
    # and should NOT be overwritten with normalized data.
    # The 'normalize_data' function already sets adata.raw to a copy of the filtered raw counts,
    # so this line below is incorrect and causes the issue.
    # adata.raw = adata # <--- REMOVING THIS LINE
    adata = adata[:, adata.var['highly_variable']]

    return adata

In [ ]:
# Select HVGs
adata = select_highly_variable_genes(adata, n_top_genes=2000)

## Data Scaling and Regression

In [ ]:
def scale_data(adata,
              vars_to_regress=None,
              max_value=10,
              use_gpu=False):
    """
    Scale data and optionally regress out unwanted sources of variation

    Parameters:
    -----------
    vars_to_regress : list
        Variables to regress out (e.g., ['total_counts', 'pct_counts_mt'])
    max_value : float
        Clip values exceeding this threshold
    use_gpu : bool
        Use GPU acceleration if available
    """

    if use_gpu and GPU_AVAILABLE:
        import rapids_singlecell as rsc
        rsc.pp.scale(adata, max_value=max_value)

        if vars_to_regress:
            rsc.pp.regress_out(adata, vars_to_regress)
    else:
        # Scale to unit variance
        sc.pp.scale(adata, max_value=max_value)

        # Regress out unwanted sources of variation
        if vars_to_regress:
            sc.pp.regress_out(adata, vars_to_regress)

    print(f"Data scaled. Max value: {max_value}")
    if vars_to_regress:
        print(f"Regressed out: {vars_to_regress}")

    return adata

In [ ]:
adata.obs

In [ ]:
# Scale data
adata = scale_data(adata, vars_to_regress=['total_counts', 'pct_counts_mt'])

## Dimensionality Reduction


### Principal Component Analysis (PCA)

In [ ]:
%%capture
!pip install kneed

In [ ]:
def run_pca(adata,
           n_comps=50,
           use_hvg=True,
           svd_solver='auto',
           random_state=0):
    """
    Run PCA with optimal component selection

    Parameters:
    -----------
    n_comps : int
        Number of principal components
    use_hvg : bool
        Use only highly variable genes
    svd_solver : str
        SVD solver to use ('auto', 'full', 'arpack', 'randomized')
    """

    # Run PCA
    sc.tl.pca(adata, n_comps=n_comps, svd_solver=svd_solver, random_state=random_state)

    # Determine optimal number of PCs
    # Method 1: Elbow plot
    fig, axes = plt.subplots(1, 3, figsize=(18, 5)) # Adjusted figsize for better display of 3 plots

    # Variance explained
    axes[0].plot(range(1, n_comps+1), adata.uns['pca']['variance_ratio'], 'o-')
    axes[0].set_xlabel('Principal Component')
    axes[0].set_ylabel('Variance Explained Ratio')
    axes[0].set_title('PCA Variance Explained')

    # Cumulative variance
    cumsum_var = np.cumsum(adata.uns['pca']['variance_ratio'])
    axes[1].plot(range(1, n_comps+1), cumsum_var, 'o-')
    axes[1].axhline(y=0.9, color='r', linestyle='--', label='90% variance')
    axes[1].set_xlabel('Principal Component')
    axes[1].set_ylabel('Cumulative Variance Explained')
    axes[1].set_title('Cumulative Variance')
    axes[1].legend()

    # Find elbow point (simplified)
    from kneed import KneeLocator # Ensure kneed is imported
    try:
        kn = KneeLocator(range(1, n_comps+1),
                        adata.uns['pca']['variance_ratio'],
                        curve='convex',
                        direction='decreasing')
        elbow = kn.elbow
        axes[0].axvline(x=elbow, color='r', linestyle='--', label=f'Elbow at PC{elbow}')
        axes[0].legend()
        print(f"Suggested number of PCs (elbow method): {elbow}")
    except Exception as e:
        print(f"Could not determine elbow point automatically: {e}")

    # Method 2: Plot PCA scatter plot (cells in PC space) on axes[2]
    # Replaced sc.pl.pca_loadings with sc.pl.pca as pca_loadings does not support the 'ax' argument
    # in this scanpy version, but sc.pl.pca does, maintaining the multi-panel figure.
    # Note: This changes the type of plot in the third panel from gene loadings to cell embeddings.
    sc.pl.pca(adata, ax=axes[2], show=False, title='PCA (Cells in PC space)')

    plt.tight_layout()
    plt.show()

    # Additional visualization
    sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

    return adata

In [ ]:
# Run PCA
adata = run_pca(adata, n_comps=50)

### UMAP and t-SNE

In [ ]:
def run_umap_tsne(adata,
                 n_neighbors=30,
                 n_pcs=40,
                 min_dist=0.3,
                 spread=1.0,
                 perplexity=30,
                 learning_rate=200,
                 use_gpu=False):
    """
    Run UMAP and t-SNE with parameter optimization

    Parameters:
    -----------
    n_neighbors : int
        Number of neighbors for UMAP
    n_pcs : int
        Number of PCs to use
    min_dist : float
        Minimum distance for UMAP
    spread : float
        Spread parameter for UMAP
    perplexity : int
        Perplexity for t-SNE
    """

    # Compute neighbor graph
    print(f"Computing neighbor graph with {n_neighbors} neighbors...")
    if use_gpu and GPU_AVAILABLE:
        import rapids_singlecell as rsc
        rsc.pp.neighbors(adata, n_neighbors=n_neighbors, n_pcs=n_pcs)
    else:
        sc.pp.neighbors(adata, n_neighbors=n_neighbors, n_pcs=n_pcs)

    # Run UMAP
    print("Running UMAP...")
    if use_gpu and GPU_AVAILABLE:
        import rapids_singlecell as rsc
        rsc.tl.umap(adata, min_dist=min_dist, spread=spread)
    else:
        sc.tl.umap(adata, min_dist=min_dist, spread=spread)

    # Run t-SNE
    print("Running t-SNE...")
    if use_gpu and GPU_AVAILABLE:
        import rapids_singlecell as rsc
        rsc.tl.tsne(adata, perplexity=perplexity, learning_rate=learning_rate)
    else:
        sc.tl.tsne(adata, perplexity=perplexity, learning_rate=learning_rate)

    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    sc.pl.umap(adata, ax=axes[0], show=False)
    sc.pl.tsne(adata, ax=axes[1], show=False)
    plt.tight_layout()
    plt.show()

    return adata

In [ ]:
# Run dimensionality reduction (UMAP)
adata = run_umap_tsne(adata, n_neighbors=30, n_pcs=40)

### Diffusion Maps (Alternative)

In [ ]:
def run_diffusion_map(adata, n_comps=15):
    """
    Run diffusion map for trajectory analysis
    """
    # Compute diffusion map
    sc.tl.diffmap(adata, n_comps=n_comps)

    # Plot
    sc.pl.diffmap(adata, color='n_genes', components=['1,2', '1,3'])

    return adata

## Clustering Analysis


### Leiden Clustering with Resolution Optimization

In [ ]:
def optimize_clustering_resolution(adata,
                                  resolution_range=(0.1, 2.0, 0.1),
                                  n_neighbors=30,
                                  metric='euclidean',
                                  random_state=0):
    """
    Optimize clustering resolution using multiple metrics

    Parameters:
    -----------
    resolution_range : tuple
        (start, stop, step) for resolution search
    """
    import pandas as pd
    from sklearn import metrics

    resolutions = np.arange(*resolution_range)

    # Store metrics
    results = []

    for res in resolutions:
        # Run clustering
        sc.tl.leiden(adata, resolution=res, random_state=random_state, key_added=f'leiden_{res}')

        # Calculate metrics
        labels = adata.obs[f'leiden_{res}'].astype(int)
        n_clusters = len(np.unique(labels))

        # Silhouette score (subsampled for speed)
        if adata.n_obs > 5000:
            subsample_idx = np.random.choice(adata.n_obs, 5000, replace=False)
            sil_score = metrics.silhouette_score(
                adata.obsm['X_pca'][subsample_idx],
                labels[subsample_idx]
            )
        else:
            sil_score = metrics.silhouette_score(adata.obsm['X_pca'], labels)

        # Calinski-Harabasz score
        ch_score = metrics.calinski_harabasz_score(adata.obsm['X_pca'], labels)

        # Davies-Bouldin score (lower is better)
        db_score = metrics.davies_bouldin_score(adata.obsm['X_pca'], labels)

        results.append({
            'resolution': res,
            'n_clusters': n_clusters,
            'silhouette': sil_score,
            'calinski_harabasz': ch_score,
            'davies_bouldin': db_score
        })

    results_df = pd.DataFrame(results)

    # Plot metrics
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    axes[0, 0].plot(results_df['resolution'], results_df['n_clusters'], 'o-')
    axes[0, 0].set_xlabel('Resolution')
    axes[0, 0].set_ylabel('Number of Clusters')
    axes[0, 0].set_title('Clusters vs Resolution')

    axes[0, 1].plot(results_df['resolution'], results_df['silhouette'], 'o-')
    axes[0, 1].set_xlabel('Resolution')
    axes[0, 1].set_ylabel('Silhouette Score')
    axes[0, 1].set_title('Silhouette Score (higher is better)')

    axes[1, 0].plot(results_df['resolution'], results_df['calinski_harabasz'], 'o-')
    axes[1, 0].set_xlabel('Resolution')
    axes[1, 0].set_ylabel('Calinski-Harabasz Score')
    axes[1, 0].set_title('Calinski-Harabasz Score (higher is better)')

    axes[1, 1].plot(results_df['resolution'], results_df['davies_bouldin'], 'o-')
    axes[1, 1].set_xlabel('Resolution')
    axes[1, 1].set_ylabel('Davies-Bouldin Score')
    axes[1, 1].set_title('Davies-Bouldin Score (lower is better)')

    plt.tight_layout()
    plt.show()

    # Find optimal resolution (based on silhouette score)
    optimal_res = results_df.loc[results_df['silhouette'].idxmax(), 'resolution']
    print(f"\nOptimal resolution (by silhouette score): {optimal_res}")
    print(f"Number of clusters: {results_df.loc[results_df['resolution']==optimal_res, 'n_clusters'].values[0]}")

    # Set final clustering
    adata.obs['leiden'] = adata.obs[f'leiden_{optimal_res}']

    # Clean up temporary clusterings
    for res in resolutions:
        del adata.obs[f'leiden_{res}']

    return adata, results_df

In [ ]:
# Optimize clustering
adata, clustering_metrics = optimize_clustering_resolution(adata)

In [ ]:
adata

In [ ]:
clustering_metrics

### Alternative Clustering Methods

In [ ]:
%%capture
!pip uninstall -y louvain igraph python-igraph leidenalg

In [ ]:
%%capture
conda remove --force louvain igraph python-igraph leidenalg

In [ ]:
%%capture
!pip install python-igraph~=0.11 louvain leidenalg

In [ ]:
%%capture
conda install -c conda-forge scanpy louvain python-igraph leidenalg

In [ ]:
def compare_clustering_algorithms(adata, resolution=1.0):
    """
    Compare different clustering algorithms
    """

    # Leiden
    sc.tl.leiden(adata, resolution=resolution, key_added='leiden')

    # Louvain
    sc.tl.louvain(adata, resolution=resolution, key_added='louvain')

    # Spectral clustering
    from sklearn.cluster import SpectralClustering
    spec_clustering = SpectralClustering(
        n_clusters=len(adata.obs['leiden'].unique()),
        affinity='precomputed',
        random_state=0
    )
    connectivity = adata.obsp['connectivities'].toarray()
    adata.obs['spectral'] = spec_clustering.fit_predict(connectivity).astype(str)

    # Visualize all methods
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    sc.pl.umap(adata, color='leiden', ax=axes[0], show=False, title='Leiden')
    sc.pl.umap(adata, color='louvain', ax=axes[1], show=False, title='Louvain')
    sc.pl.umap(adata, color='spectral', ax=axes[2], show=False, title='Spectral')
    plt.tight_layout()
    plt.show()

    return adata

In [ ]:
adata

In [ ]:
# Compare clustering methods
adata = compare_clustering_algorithms(adata)

## Differential Expression Analysis


### Finding Marker Genes

In [ ]:
def find_all_markers(adata,
                    groupby='leiden',
                    method='wilcoxon',
                    min_fold_change=0.25,
                    min_pct=0.1,
                    n_genes=25):
    """
    Find marker genes for all clusters

    Parameters:
    -----------
    method : str
        'wilcoxon', 't-test', 'logreg', or 'wilcoxon_pval'
    min_fold_change : float
        Minimum log fold change
    min_pct : float
        Minimum percentage of cells expressing the gene
    """

    # Calculate marker genes
    print(f"Finding markers using {method} test...")
    sc.tl.rank_genes_groups(
        adata,
        groupby=groupby,
        method=method,
        min_fold_change=min_fold_change,
        n_genes=n_genes,
        use_raw=True
    )

    # Filter by expression percentage
    sc.tl.filter_rank_genes_groups(
        adata,
        min_in_group_fraction=min_pct,
        min_fold_change=min_fold_change
    )

    # Visualize top markers
    sc.pl.rank_genes_groups(adata, n_genes=n_genes, sharey=False)

    # Create marker gene DataFrame
    markers_df = pd.DataFrame()
    for cluster in adata.obs[groupby].unique():
        cluster_genes = pd.DataFrame({
            'gene': adata.uns['rank_genes_groups']['names'][cluster][:n_genes],
            'score': adata.uns['rank_genes_groups']['scores'][cluster][:n_genes],
            'logfoldchange': adata.uns['rank_genes_groups']['logfoldchanges'][cluster][:n_genes],
            'pval': adata.uns['rank_genes_groups']['pvals'][cluster][:n_genes],
            'pval_adj': adata.uns['rank_genes_groups']['pvals_adj'][cluster][:n_genes],
            'cluster': cluster
        })
        markers_df = pd.concat([markers_df, cluster_genes])

    # Save markers
    markers_df.to_csv('marker_genes.csv', index=False)
    print(f"Saved marker genes to marker_genes.csv")

    return adata, markers_df

In [ ]:
# Find markers
adata, markers = find_all_markers(adata)

In [ ]:
adata

In [ ]:
markers

### Visualization of Marker Genes

In [ ]:
def visualize_markers(adata, markers_df, n_top=5):
    """
    Create comprehensive marker gene visualizations
    """

    # Get top markers per cluster
    top_markers = markers_df.groupby('cluster').head(n_top)['gene'].unique()

    # Dot plot
    sc.pl.dotplot(adata, top_markers, groupby='leiden',
                  dendrogram=True, standard_scale='var')

    # Heatmap
    sc.pl.heatmap(adata, top_markers, groupby='leiden',
                  swap_axes=True, standard_scale='var', cmap='RdBu_r')

    # Stacked violin plot
    sc.pl.stacked_violin(adata, top_markers, groupby='leiden',
                         dendrogram=True)

    # Matrix plot
    sc.pl.matrixplot(adata, top_markers, groupby='leiden',
                     dendrogram=True, standard_scale='var', cmap='RdBu_r')

    return adata

In [ ]:
# Visualize markers
adata = visualize_markers(adata, markers)

## Cell Type Annotation


###  Manual Annotation with Known Markers

In [ ]:
def manual_annotation(adata, marker_dict, groupby='leiden'):
    """
    Manually annotate cell types based on known markers

    Parameters:
    -----------
    marker_dict : dict
        Dictionary with cell types as keys and marker lists as values
    """

    # Example marker dictionary for PBMCs
    if marker_dict is None:
        marker_dict = {
            'CD4 T cells': ['CD3D', 'CD4', 'IL7R'],
            'CD8 T cells': ['CD3D', 'CD8A', 'CD8B'],
            'NK cells': ['GNLY', 'NKG7', 'KLRD1'],
            'B cells': ['MS4A1', 'CD79A', 'CD79B'],
            'Monocytes': ['CD14', 'LYZ', 'S100A8', 'S100A9'],
            'DCs': ['FCER1A', 'CST3', 'CLEC10A'],
            'Platelets': ['PPBP', 'PF4']
        }

    # Score each cell type
    for cell_type, markers in marker_dict.items():
        # Filter markers that exist in the dataset
        valid_markers = [m for m in markers if m in adata.var_names]
        if valid_markers:
            sc.tl.score_genes(adata, valid_markers, score_name=f'{cell_type}_score')

    # Create score matrix
    score_cols = [f'{ct}_score' for ct in marker_dict.keys()]
    score_matrix = adata.obs[score_cols].values

    # Assign cell types based on highest score per cluster
    cluster_annotations = {}
    for cluster in adata.obs[groupby].unique():
        cluster_mask = adata.obs[groupby] == cluster
        cluster_scores = score_matrix[cluster_mask].mean(axis=0)
        best_type = list(marker_dict.keys())[np.argmax(cluster_scores)]
        cluster_annotations[cluster] = best_type

    # Add annotations
    adata.obs['cell_type'] = adata.obs[groupby].map(cluster_annotations)

    # Visualize
    sc.pl.umap(adata, color=['leiden', 'cell_type'], legend_loc='on data')

    return adata

In [ ]:
adata

In [ ]:
def manual_annotation(adata, marker_dict, groupby='leiden'):
    """
    Manually annotate cell types based on known markers

    Parameters:
    -----------
    marker_dict : dict
        Dictionary with cell types as keys and marker lists as values
    """

    if marker_dict is None:
        # This block is usually for default, but for this context, it's explicitly passed.
        # Keeping it for consistency with original function structure.
        marker_dict = {
            'CD4 T cells': ['CD3D', 'CD4', 'IL7R'],
            'CD8 T cells': ['CD8A', 'CD8B'],
            'NK cells': ['GNLY', 'NKG7', 'KLRD1'],
            'B cells': ['MS4A1', 'CD79A', 'CD79B'],
            'Monocytes': ['CD14', 'LYZ', 'S100A8', 'S100A9'],
            'DCs': ['FCER1A', 'CST3', 'CLEC10A'],
            'Platelets': ['PPBP', 'PF4']
        }

    actual_score_cols = []
    cell_types_with_scores = []

    # Score each cell type
    for cell_type, markers in marker_dict.items():
        # Filter markers that exist in the dataset
        valid_markers = [m for m in markers if m in adata.var_names]
        if valid_markers:
            score_name = f'{cell_type}_score'
            sc.tl.score_genes(adata, valid_markers, score_name=score_name)
            actual_score_cols.append(score_name)
            cell_types_with_scores.append(cell_type)
        else:
            print(f"Warning: No valid markers found for '{cell_type}' among the provided genes. Skipping scoring for this cell type.")

    # Check if any scores were successfully calculated
    if not actual_score_cols:
        print("Error: No cell type scores could be calculated. Please check your marker dictionary and ensure markers are present in adata.var_names.")
        # Add a placeholder 'cell_type' column to prevent potential downstream errors
        adata.obs['cell_type'] = 'Unassigned'
        return adata

    # Create score matrix using only the actually created columns
    score_matrix = adata.obs[actual_score_cols].values

    # Assign cell types based on highest score per cluster
    cluster_annotations = {}
    for cluster in adata.obs[groupby].unique():
        cluster_mask = adata.obs[groupby] == cluster
        # Ensure we're using only the calculated scores for this cluster
        cluster_scores = adata.obs.loc[cluster_mask, actual_score_cols].values.mean(axis=0)

        if len(cluster_scores) > 0:
            best_type_index = np.argmax(cluster_scores)
            best_type = cell_types_with_scores[best_type_index]
            cluster_annotations[cluster] = best_type
        else:
            cluster_annotations[cluster] = 'Unassigned' # Fallback if no scores for a cluster

    # Add annotations
    adata.obs['cell_type'] = adata.obs[groupby].map(cluster_annotations).astype('category')

    # Visualize
    # Only visualize if UMAP coordinates exist
    if 'X_umap' in adata.obsm_keys():
        import matplotlib.pyplot as plt
        sc.pl.umap(adata, color=['leiden', 'cell_type'], legend_loc='on data', show=False)
        plt.show() # Explicitly show the plot
    else:
        print("Warning: UMAP coordinates not found (X_umap in adata.obsm). Skipping UMAP visualization.")

    return adata

In [ ]:
# Annotate cell types
# Define a marker dictionary with mouse genes
mouse_marker_dict = {
    'CD4 T cells': ['Cd4', 'Cd3e', 'Il7r'],
    'CD8 T cells': ['Cd8a', 'Cd3e'],
    'NK cells': ['Nkg7', 'Klrd1'],
    'B cells': ['Cd79a', 'Cd19'],
    'Monocytes': ['Cd14', 'Lyz2'],
    'DCs': ['Fcer1a', 'Itgax'],
    'Platelets': ['Pf4']
}
adata = manual_annotation(adata, marker_dict=mouse_marker_dict)

## Automated Annotation with CellTypist

In [ ]:
%%capture
!pip install celltypist

In [ ]:
import celltypist

In [ ]:
print(celltypist.__version__)

In [ ]:
def automated_annotation_celltypist(
    adata,
    model: str = "Immune_All_Low.pkl",
    majority_voting: bool = True,
    label_key: str = "celltypist_prediction",
    confidence_key: str = "celltypist_confidence"
):
    """
    Version-robust automated cell type annotation using CellTypist.
    Works across CellTypist releases with or without `confidence` attribute.
    """
    import scanpy as sc
    import celltypist
    from celltypist import models

    # 1. Preconditions
    if adata.raw is None:
        raise ValueError("adata.raw must contain raw counts.")

    # 2. Load model
    models.download_models(force_update=False)
    ct_model = models.Model.load(model=model)

    # 3. Prepare data (CP10K + log1p)
    adata_ct = adata.raw.to_adata()
    sc.pp.normalize_total(adata_ct, target_sum=1e4)
    sc.pp.log1p(adata_ct)

    # 4. Run CellTypist
    predictions = celltypist.annotate(
        adata_ct,
        model=ct_model,
        majority_voting=majority_voting
    )

    # 5. Extract predicted labels (safe for all versions)
    labels = predictions.predicted_labels.iloc[:, 0]
    labels = labels.reindex(adata.obs_names)
    adata.obs[label_key] = labels.astype("category")

    # 6. Derive confidence robustly
    prob = predictions.probability_matrix
    prob = prob.reindex(adata.obs_names)

    confidence = prob.max(axis=1)
    adata.obs[confidence_key] = confidence

    # 7. Optional visualization
    if "X_umap" in adata.obsm:
        sc.pl.umap(
            adata,
            color=[label_key, confidence_key],
            legend_loc="on data",
            frameon=False
        )

    # 8. Sanity check
    assert adata.n_obs == adata.obs[label_key].shape[0]

    return adata

In [ ]:
adata = automated_annotation_celltypist(adata, model="Immune_All_Low.pkl")

In [ ]:
adata.obs

In [ ]:
# Check if celltypist_prediction exists in adata.obs
if 'celltypist_prediction' in adata.obs.columns:
	print(adata.obs['celltypist_prediction'].value_counts())
else:
	print("'celltypist_prediction' column not found in adata.obs.")
	print("Available columns:", list(adata.obs.columns))
	print("\nNote: CellTypist models are trained on human genes. Your data appears to contain mouse genes.")
	print("Consider converting gene names to human orthologs or using a mouse-specific model.")

In [ ]:
adata.obs['cell_type'].value_counts()

### Mouse-Specific Cell Type Annotation

Since CellTypist is trained on human genes, we'll use mouse-specific marker genes for annotation.

In [ ]:
def annotate_mouse_cells(adata, marker_dict=None, groupby='leiden', method='overlap'):
    """
    Annotate mouse single-cell data using marker gene-based scoring.
    
    Parameters:
    -----------
    adata : AnnData
        Annotated data matrix with mouse genes
    marker_dict : dict
        Dictionary with cell types as keys and list of marker genes as values.
        If None, uses default mouse brain/neural markers.
    groupby : str
        Clustering key in adata.obs to use for annotation
    method : str
        'overlap' - Use marker gene overlap scoring
        'score' - Use scanpy's score_genes method
        
    Returns:
    --------
    adata : AnnData with 'mouse_cell_type' column added
    """
    
    # Default mouse brain/neural marker genes
    if marker_dict is None:
        marker_dict = {
            # Neurons
            'Excitatory_Neurons': ['Slc17a7', 'Slc17a6', 'Neurod6', 'Satb2', 'Tbr1', 'Camk2a'],
            'Inhibitory_Neurons': ['Gad1', 'Gad2', 'Slc32a1', 'Dlx1', 'Dlx2', 'Dlx5'],
            'Dopaminergic_Neurons': ['Th', 'Slc6a3', 'Ddc', 'Nr4a2', 'Pitx3'],
            'Serotonergic_Neurons': ['Tph2', 'Slc6a4', 'Fev', 'Pet1'],
            'Cholinergic_Neurons': ['Chat', 'Slc18a3', 'Ache'],
            
            # Glia
            'Astrocytes': ['Gfap', 'Aqp4', 'Aldh1l1', 'S100b', 'Slc1a3', 'Slc1a2'],
            'Oligodendrocytes': ['Mbp', 'Mog', 'Plp1', 'Olig1', 'Olig2', 'Sox10'],
            'OPCs': ['Pdgfra', 'Cspg4', 'Olig2', 'Sox10'],
            'Microglia': ['Cx3cr1', 'P2ry12', 'Tmem119', 'Hexb', 'Aif1', 'Itgam'],
            
            # Other
            'Endothelial': ['Pecam1', 'Cldn5', 'Flt1', 'Vwf', 'Cdh5'],
            'Pericytes': ['Pdgfrb', 'Rgs5', 'Kcnj8', 'Abcc9'],
            'Fibroblasts': ['Col1a1', 'Col1a2', 'Dcn', 'Lum'],
            
            # Neural progenitors
            'Neural_Progenitors': ['Nes', 'Sox2', 'Pax6', 'Hes5', 'Notch1'],
            'Radial_Glia': ['Vim', 'Fabp7', 'Slc1a3', 'Pax6', 'Hes1'],
            
            # Immune (if present in brain tissue)
            'T_cells': ['Cd3d', 'Cd3e', 'Cd3g', 'Trac'],
            'B_cells': ['Cd19', 'Cd79a', 'Cd79b', 'Ms4a1'],
            'Macrophages': ['Cd68', 'Adgre1', 'Mrc1', 'Csf1r'],
        }
    
    # Get available genes in dataset
    available_genes = set(adata.var_names)
    
    # Filter marker dict to only include available genes
    filtered_markers = {}
    for cell_type, markers in marker_dict.items():
        available_markers = [g for g in markers if g in available_genes]
        if available_markers:
            filtered_markers[cell_type] = available_markers
            print(f"{cell_type}: {len(available_markers)}/{len(markers)} markers found")
    
    if not filtered_markers:
        print("Warning: No marker genes found in dataset!")
        print("Check if gene names match (e.g., capitalization)")
        return adata
    
    if method == 'score':
        # Use scanpy's score_genes for each cell type
        for cell_type, markers in filtered_markers.items():
            sc.tl.score_genes(adata, markers, score_name=f'{cell_type}_score')
        
        # Assign cell type based on highest score
        score_cols = [f'{ct}_score' for ct in filtered_markers.keys()]
        scores_df = adata.obs[score_cols]
        adata.obs['mouse_cell_type'] = scores_df.idxmax(axis=1).str.replace('_score', '')
        
    elif method == 'overlap':
        # Overlap-based scoring per cluster
        cluster_annotations = {}
        
        for cluster in adata.obs[groupby].unique():
            cluster_mask = adata.obs[groupby] == cluster
            cluster_data = adata[cluster_mask]
            
            # Get highly expressed genes in this cluster
            if hasattr(cluster_data.X, 'toarray'):
                mean_expr = np.array(cluster_data.X.toarray().mean(axis=0)).flatten()
            else:
                mean_expr = np.array(cluster_data.X.mean(axis=0)).flatten()
            
            # Get top expressed genes
            top_genes_idx = np.argsort(mean_expr)[-200:]
            top_genes = set(adata.var_names[top_genes_idx])
            
            # Score each cell type by marker overlap
            best_score = 0
            best_type = 'Unknown'
            
            for cell_type, markers in filtered_markers.items():
                overlap = len(top_genes.intersection(markers))
                score = overlap / len(markers)  # Normalize by number of markers
                if score > best_score:
                    best_score = score
                    best_type = cell_type
            
            cluster_annotations[cluster] = best_type
            print(f"Cluster {cluster} -> {best_type} (score: {best_score:.2f})")
        
        # Map annotations to cells
        adata.obs['mouse_cell_type'] = adata.obs[groupby].map(cluster_annotations)
    
    # Visualize results
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    sc.pl.umap(adata, color=groupby, ax=axes[0], show=False, 
               title=f'Clusters ({groupby})', frameon=False)
    sc.pl.umap(adata, color='mouse_cell_type', ax=axes[1], show=False,
               title='Mouse Cell Type Annotation', frameon=False)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print("\n--- Cell Type Distribution ---")
    print(adata.obs['mouse_cell_type'].value_counts())
    
    return adata

In [ ]:
# Run mouse-specific cell type annotation
adata = annotate_mouse_cells(adata, groupby='leiden', method='overlap')

In [ ]:
# View the mouse cell type annotations
adata.obs['mouse_cell_type'].value_counts()

In [ ]:
adata.n_obs

In [ ]:
# Example usage (requires celltypist installation)
adata = automated_annotation_celltypist(adata)

## Trajectory Analysis


### Pseudotime Analysis with Diffusion Pseudotime

In [ ]:
def calculate_pseudotime(adata, root_cluster='0', groupby='leiden'):
    """
    Calculate pseudotime using diffusion maps

    Parameters:
    -----------
    root_cluster : str
        Cluster to use as root for pseudotime
    """

    # Calculate diffusion map if not already done
    if 'X_diffmap' not in adata.obsm:
        sc.tl.diffmap(adata)

    # Set root cell (cell with highest expression of root markers)
    root_mask = adata.obs[groupby] == root_cluster
    root_indices = np.where(root_mask)[0]

    # Use the cell closest to the median of the root cluster
    root_cell = root_indices[0]
    adata.uns['iroot'] = root_cell

    # Calculate diffusion pseudotime
    sc.tl.dpt(adata)

    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    sc.pl.umap(adata, color=groupby, ax=axes[0], show=False, title='Clusters')
    sc.pl.umap(adata, color='dpt_pseudotime', ax=axes[1], show=False, title='Pseudotime')
    sc.pl.diffmap(adata, color='dpt_pseudotime', ax=axes[2], show=False)
    plt.tight_layout()
    plt.show()

    return adata

In [ ]:
# Calculate pseudotime
adata = calculate_pseudotime(adata, root_cluster='0')

In [ ]:
adata

### RNA Velocity Analysis

In [ ]:
def run_velocity_analysis(adata, loom_file=None):
    """
    Run RNA velocity analysis using scVelo

    Parameters:
    -----------
    loom_file : str
        Path to loom file with spliced/unspliced counts
    """
    import scvelo as scv

    if loom_file:
        # Load velocity data
        ldata = scv.read(loom_file, cache=True)

        # Merge with adata
        adata = scv.utils.merge(adata, ldata)

    # Preprocess
    scv.pp.filter_and_normalize(adata, min_shared_counts=30, n_top_genes=2000)
    scv.pp.moments(adata, n_pcs=30, n_neighbors=30)

    # Run velocity
    scv.tl.velocity(adata)
    scv.tl.velocity_graph(adata)

    # Project velocity
    scv.pl.velocity_embedding_stream(adata, basis='umap', color='leiden')

    # Calculate velocity confidence
    scv.tl.velocity_confidence(adata)

    # Visualize
    scv.pl.velocity_embedding(adata, arrow_length=3, arrow_size=2, dpi=120)

    return adata

In [ ]:
# Example usage (requires loom file)
adata = run_velocity_analysis(adata, loom_file='velocyto_output.loom')